# Advanced `ChainMap` — Tutorial-Style Problems with Guided Solutions

This notebook is a **second, independent advanced practice notebook** on `collections.ChainMap`.

The style is deliberately tutorial-oriented:

- introduce one question at a time,
- make a prediction,
- run a small experiment,
- explain what happened,
- then combine the idea into a harder problem.

The goal is not just to get correct answers, but to build a reliable mental model for layered mappings.

### What we will build toward

We will move from small observations to advanced applications:

1. precedence and collisions,
2. shadowing and unshadowing,
3. live-reference behavior,
4. controlled writes,
5. scope stacks,
6. configuration explainability,
7. temporary overrides,
8. validation,
9. custom mutation policies,
10. audit/debug tools,
11. nested environments,
12. transactional overlays.

We will use assertions frequently so that each solution also acts as an executable test.

In [1]:
from collections import ChainMap
from collections.abc import MutableMapping
from types import MappingProxyType
from pprint import pprint

print("Ready.")

Ready.


# 1. First, let's revisit precedence carefully

Suppose we have three mappings.

The same key appears in more than one mapping.

Before running the next cell, answer this:

> If a key exists in all three maps, which value does a `ChainMap` return?

In [2]:
d1 = {"x": 1, "shared": "first"}
d2 = {"y": 2, "shared": "second"}
d3 = {"z": 3, "shared": "third"}

cm = ChainMap(d1, d2, d3)

The lookup order follows the order of the mappings inside the chain.

Let's verify that with one lookup.

In [3]:
cm["shared"]

'first'

The value came from `d1`.

So the simplest rule is:

> **The first matching key wins.**

Now let's turn that simple rule into a problem.

## Problem 1 — Predict an entire layered mapping

Given these mappings:

```python
request = {"timeout": 2, "trace": True}
user = {"theme": "dark", "timeout": 10}
environment = {"region": "eu", "timeout": 20}
defaults = {"theme": "light", "region": "us", "timeout": 30, "retries": 3}
```

Create a `ChainMap` with the precedence:

`request -> user -> environment -> defaults`

Without flattening it first, determine the effective values of:

- `timeout`
- `trace`
- `theme`
- `region`
- `retries`

### Solution — Step 1: construct the chain

In [4]:
request = {"timeout": 2, "trace": True}
user = {"theme": "dark", "timeout": 10}
environment = {"region": "eu", "timeout": 20}
defaults = {"theme": "light", "region": "us", "timeout": 30, "retries": 3}

config = ChainMap(request, user, environment, defaults)

### Solution — Step 2: test each lookup independently

This is intentionally verbose.

When learning layered mappings, it is useful to inspect each important lookup separately rather than immediately converting everything to a dictionary.

In [5]:
print("timeout:", config["timeout"])
print("trace:", config["trace"])
print("theme:", config["theme"])
print("region:", config["region"])
print("retries:", config["retries"])

timeout: 2
trace: True
theme: dark
region: eu
retries: 3


### Solution — Step 3: encode the expectations as tests

In [6]:
assert config["timeout"] == 2
assert config["trace"] is True
assert config["theme"] == "dark"
assert config["region"] == "eu"
assert config["retries"] == 3

print("All precedence checks passed.")

All precedence checks passed.


# 2. A lookup works, but where did the value come from?

A normal lookup tells us the final value.

In a large configuration stack, that is sometimes not enough.

We may also want to know:

> Which underlying mapping supplied that value?

## Problem 2 — Trace the source of a lookup

Write a function:

```python
def trace_lookup(cm, key):
    ...
```

It should return a dictionary containing:

- the key,
- the visible value,
- the index of the first mapping containing it,
- every mapping index where the key appears.

If the key is absent, raise `KeyError`.

Let's solve this in small steps.

### Step 1: inspect `cm.maps`

The `maps` attribute gives direct access to the mappings in lookup order.

In [7]:
config.maps

[{'timeout': 2, 'trace': True},
 {'theme': 'dark', 'timeout': 10},
 {'region': 'eu', 'timeout': 20},
 {'theme': 'light', 'region': 'us', 'timeout': 30, 'retries': 3}]

### Step 2: find all occurrences of a key

We can scan the maps from left to right.

In [8]:
key = "timeout"

occurrences = []
for index, mapping in enumerate(config.maps):
    if key in mapping:
        occurrences.append(index)

occurrences

[0, 1, 2, 3]

Since lookup uses the first match, the first item in `occurrences` is the source index.

Now we can turn that reasoning into a reusable function.

In [9]:
def trace_lookup(cm, key):
    occurrences = [
        index
        for index, mapping in enumerate(cm.maps)
        if key in mapping
    ]

    if not occurrences:
        raise KeyError(key)

    source_index = occurrences[0]

    return {
        "key": key,
        "value": cm[key],
        "source_index": source_index,
        "present_in": occurrences,
    }

Let's test a heavily shadowed key.

In [10]:
trace_lookup(config, "timeout")

{'key': 'timeout', 'value': 2, 'source_index': 0, 'present_in': [0, 1, 2, 3]}

And now a key that exists only in the defaults.

In [11]:
trace_lookup(config, "retries")

{'key': 'retries', 'value': 3, 'source_index': 3, 'present_in': [3]}

Finally, let's test the failure case.

In [12]:
try:
    trace_lookup(config, "missing")
except KeyError as exc:
    print("Expected:", exc)

Expected: 'missing'


# 3. Now let's look at writes

A very important behavior of `ChainMap` is that a normal assignment does **not** search for the mapping that already owns the key.

Instead, assignment targets the first mapping.

Let's build that behavior from a small experiment.

In [13]:
front = {"a": 1}
back = {"mode": "production", "timeout": 30}

settings = ChainMap(front, back)

print("Before:")
pprint(settings.maps)

Before:
[{'a': 1}, {'mode': 'production', 'timeout': 30}]


Suppose we assign a new value to `timeout`.

At the moment, `timeout` exists only in the second mapping.

What do you expect will happen?

In [14]:
settings["timeout"] = 5

print("After:")
pprint(settings.maps)

After:
[{'a': 1, 'timeout': 5}, {'mode': 'production', 'timeout': 30}]


The second dictionary was not edited.

A new `timeout` key appeared in the first mapping.

That new value now shadows the parent value.

## Problem 3 — Build a temporary override layer

You have a large base configuration that you do not want to modify.

Create a `ChainMap` that lets you make temporary changes to:

- `host`
- `timeout`
- a brand-new key named `debug`

Requirements:

1. the original base dictionary must remain unchanged;
2. all writes must go into a small overlay dictionary;
3. after discarding the overlay, the original configuration must be visible again.

### Solution — Step 1: create an empty map in front

In [15]:
base = {
    "host": "prod.example.com",
    "port": 443,
    "timeout": 30,
}

overlay = {}
temporary = ChainMap(overlay, base)

### Step 2: perform the temporary changes

In [16]:
temporary["host"] = "localhost"
temporary["timeout"] = 1
temporary["debug"] = True

print("Effective:")
pprint(dict(temporary))

print("\nOverlay:")
pprint(overlay)

print("\nBase:")
pprint(base)

Effective:
{'debug': True, 'host': 'localhost', 'port': 443, 'timeout': 1}

Overlay:
{'debug': True, 'host': 'localhost', 'timeout': 1}

Base:
{'host': 'prod.example.com', 'port': 443, 'timeout': 30}


### Step 3: verify that the base is untouched

In [17]:
assert base == {
    "host": "prod.example.com",
    "port": 443,
    "timeout": 30,
}

assert overlay == {
    "host": "localhost",
    "timeout": 1,
    "debug": True,
}

### Step 4: discard the temporary layer

We do not need to undo every assignment manually.

We can simply stop using the child layer.

In [18]:
restored = temporary.parents

assert restored["host"] == "prod.example.com"
assert restored["timeout"] == 30
assert "debug" not in restored

pprint(dict(restored))

{'host': 'prod.example.com', 'port': 443, 'timeout': 30}


# 4. Deleting a child key can reveal a parent key

This is one of the most useful consequences of shadowing.

Consider a value that exists in both the first and second mappings.

In [19]:
child = {"color": "red"}
parent = {"color": "blue", "size": "large"}

view = ChainMap(child, parent)

print(view["color"])

red


If we delete `color` through the `ChainMap`, the deletion targets the first mapping.

The parent's value is still there.

In [20]:
del view["color"]

print("child:", child)
print("parent:", parent)
print("visible color:", view["color"])

child: {}
parent: {'color': 'blue', 'size': 'large'}
visible color: blue


## Problem 4 — Implement "reset to inherited value"

Write a function:

```python
def reset_override(cm, key):
    ...
```

It should remove a key **only if it is present in the first mapping**.

Afterward, if a parent has the same key, the parent value should become visible.

If the first mapping does not contain the key, raise a clear `KeyError`.

### Solution — Step 1: inspect only the first mapping

We should not use `if key in cm`, because that tests the entire combined view.

We specifically care about `cm.maps[0]`.

In [21]:
def reset_override(cm, key):
    if key not in cm.maps[0]:
        raise KeyError(f"{key!r} is not overridden in the first mapping")

    del cm[key]

### Step 2: test the successful case

In [22]:
base = {"theme": "light", "timeout": 30}
overrides = {"theme": "dark"}

cm = ChainMap(overrides, base)

assert cm["theme"] == "dark"

reset_override(cm, "theme")

assert cm["theme"] == "light"
assert "theme" not in overrides

print(dict(cm))

{'theme': 'light', 'timeout': 30}


### Step 3: test the error case

`timeout` is visible, but it is not overridden in the first mapping.

In [23]:
try:
    reset_override(cm, "timeout")
except KeyError as exc:
    print("Expected:", exc)

Expected: "'timeout' is not overridden in the first mapping"


# 5. `ChainMap` is a live view

The underlying mappings are incorporated by reference.

So if an underlying dictionary changes later, the `ChainMap` sees that change.

Let's verify that in stages.

In [24]:
left = {"a": 1}
right = {"b": 2}

cm = ChainMap(left, right)

print(dict(cm))

{'b': 2, 'a': 1}


Now mutate the second dictionary directly.

In [25]:
right["c"] = 3
right["b"] = 200

print(dict(cm))

{'b': 200, 'c': 3, 'a': 1}


The chain changed immediately even though we never rebuilt it.

This gives us a more advanced problem.

## Problem 5 — Live feature flags

Create three mappings:

- `session_flags`
- `user_flags`
- `global_flags`

Build a `ChainMap` using that precedence.

Then:

1. read `new_ui`;
2. change the global flag directly;
3. prove the chain sees the change;
4. add a user-level override;
5. prove the global change is now shadowed;
6. remove the user override and prove the latest global value becomes visible again.

### Solution

In [26]:
global_flags = {
    "new_ui": False,
    "beta_search": False,
}

user_flags = {}
session_flags = {}

flags = ChainMap(session_flags, user_flags, global_flags)

assert flags["new_ui"] is False

global_flags["new_ui"] = True
assert flags["new_ui"] is True

user_flags["new_ui"] = False
assert flags["new_ui"] is False

global_flags["new_ui"] = True
assert flags["new_ui"] is False

del user_flags["new_ui"]
assert flags["new_ui"] is True

print(dict(flags))

{'new_ui': True, 'beta_search': False}


# 6. `new_child()` gives us a natural scope stack

A child mapping is placed at the beginning of the chain.

This makes it useful for nested environments such as:

- interpreters,
- templating systems,
- temporary contexts,
- nested configuration scopes.

In [27]:
global_scope = ChainMap({"x": 10, "pi": 3.14159})

function_scope = global_scope.new_child()
function_scope["x"] = 20

block_scope = function_scope.new_child()
block_scope["y"] = 30

print("global:", dict(global_scope))
print("function:", dict(function_scope))
print("block:", dict(block_scope))

global: {'x': 10, 'pi': 3.14159}
function: {'x': 20, 'pi': 3.14159}
block: {'x': 20, 'pi': 3.14159, 'y': 30}


Notice that `block_scope` can see values from all outer scopes.

But writes go only to its current child mapping.

## Problem 6 — Build a tiny environment object

Create a class `Environment` with:

- `push()` — add a child scope;
- `pop()` — remove the current child scope;
- `set(name, value)` — write to current scope;
- `get(name)` — chained lookup;
- `local(name)` — test whether a name exists only in the current scope;
- `depth` — number of active mappings.

Do not allow the global scope to be popped.

### Solution — Step 1: define the container

In [28]:
class Environment:
    def __init__(self, globals=None):
        self.chain = ChainMap({} if globals is None else globals)

    @property
    def depth(self):
        return len(self.chain.maps)

    def push(self):
        self.chain = self.chain.new_child()

    def pop(self):
        if len(self.chain.maps) == 1:
            raise RuntimeError("Cannot pop the global scope")
        self.chain = self.chain.parents

    def set(self, name, value):
        self.chain[name] = value

    def get(self, name):
        return self.chain[name]

    def local(self, name):
        return name in self.chain.maps[0]

### Step 2: test shadowing

In [29]:
env = Environment({"x": 1})

assert env.depth == 1
assert env.get("x") == 1

env.push()
env.set("x", 2)
env.set("y", 3)

assert env.depth == 2
assert env.get("x") == 2
assert env.get("y") == 3
assert env.local("x")

### Step 3: add another nested scope

In [30]:
env.push()
env.set("z", 4)

assert env.depth == 3
assert env.get("x") == 2
assert env.get("y") == 3
assert env.get("z") == 4

assert not env.local("x")
assert env.local("z")

### Step 4: pop scopes and watch visibility change

In [31]:
env.pop()
assert env.depth == 2
assert env.get("x") == 2

env.pop()
assert env.depth == 1
assert env.get("x") == 1

try:
    env.get("y")
except KeyError:
    print("y is no longer visible, as expected.")

y is no longer visible, as expected.


# 7. Advanced debugging: explain every effective value

For a large configuration stack, it is often useful to produce a report like:

> `timeout = 5`, coming from layer 0, and shadowing values in layers 2 and 3.

Let's build that report gradually.

## Problem 7 — Configuration provenance report

Write:

```python
def explain_config(cm):
    ...
```

For every visible key, return:

- the effective value,
- the source index,
- all indexes where the key occurs,
- whether the key is shadowing lower-priority values.

### Step 1: collect every key occurrence

We can build a dictionary mapping each key to a list of layer indexes.

In [32]:
layers = ChainMap(
    {"timeout": 5, "debug": True},
    {"theme": "dark"},
    {"timeout": 20, "theme": "light"},
    {"timeout": 30, "retries": 3, "debug": False},
)

locations = {}

for index, mapping in enumerate(layers.maps):
    for key in mapping:
        locations.setdefault(key, []).append(index)

pprint(locations)

{'debug': [0, 3], 'retries': [3], 'theme': [1, 2], 'timeout': [0, 2, 3]}


### Step 2: convert those locations into an explanation

The first occurrence is always the visible source.

In [33]:
def explain_config(cm):
    locations = {}

    for index, mapping in enumerate(cm.maps):
        for key in mapping:
            locations.setdefault(key, []).append(index)

    result = {}

    for key, indexes in locations.items():
        result[key] = {
            "value": cm[key],
            "source_index": indexes[0],
            "present_in": indexes,
            "shadows_lower_layers": len(indexes) > 1,
        }

    return result

### Step 3: inspect the report

In [34]:
report = explain_config(layers)
pprint(report)

{'debug': {'present_in': [0, 3],
           'shadows_lower_layers': True,
           'source_index': 0,
           'value': True},
 'retries': {'present_in': [3],
             'shadows_lower_layers': False,
             'source_index': 3,
             'value': 3},
 'theme': {'present_in': [1, 2],
           'shadows_lower_layers': True,
           'source_index': 1,
           'value': 'dark'},
 'timeout': {'present_in': [0, 2, 3],
             'shadows_lower_layers': True,
             'source_index': 0,
             'value': 5}}


### Step 4: verify important cases

In [35]:
assert report["timeout"]["value"] == 5
assert report["timeout"]["source_index"] == 0
assert report["timeout"]["present_in"] == [0, 2, 3]

assert report["theme"]["value"] == "dark"
assert report["theme"]["source_index"] == 1

assert report["retries"]["source_index"] == 3

print("Report checks passed.")

Report checks passed.


# 8. Layering is useful, but configuration still needs validation

`ChainMap` handles precedence.

It does not automatically check:

- required keys,
- value types,
- allowed ranges,
- relationships between values.

So a robust application often separates:

1. **composition** of configuration,
2. **validation** of the effective result.

## Problem 8 — Validate an effective configuration

Write:

```python
def validate_server_config(cm):
    ...
```

Rules:

- `host` must exist and be a non-empty string;
- `port` must be an integer from 1 to 65535;
- `timeout` must be a positive number;
- `retries` must be an integer from 0 to 10.

Return `True` when valid.

Raise `ValueError` with a useful message otherwise.

### Solution — validate the visible values

Notice that we validate `cm[key]`, not each layer independently.

A low-priority invalid value does not matter if it is shadowed by a valid higher-priority value.

In [36]:
def validate_server_config(cm):
    required = ("host", "port", "timeout", "retries")
    missing = [key for key in required if key not in cm]

    if missing:
        raise ValueError(f"Missing required keys: {missing}")

    host = cm["host"]
    port = cm["port"]
    timeout = cm["timeout"]
    retries = cm["retries"]

    if not isinstance(host, str) or not host.strip():
        raise ValueError("host must be a non-empty string")

    if not isinstance(port, int) or isinstance(port, bool) or not 1 <= port <= 65535:
        raise ValueError("port must be an integer from 1 to 65535")

    if not isinstance(timeout, (int, float)) or isinstance(timeout, bool) or timeout <= 0:
        raise ValueError("timeout must be a positive number")

    if not isinstance(retries, int) or isinstance(retries, bool) or not 0 <= retries <= 10:
        raise ValueError("retries must be an integer from 0 to 10")

    return True

Let's test a valid layered configuration.

In [37]:
defaults = {
    "host": "api.example.com",
    "port": 443,
    "timeout": 30,
    "retries": 3,
}

environment = {"timeout": 15}
user = {"retries": 5}
request = {}

cm = ChainMap(request, user, environment, defaults)

assert validate_server_config(cm) is True
print("Valid configuration.")

Valid configuration.


Now create a temporary invalid override.

The defaults are still valid, but the visible configuration is not.

In [38]:
request["port"] = 70000

try:
    validate_server_config(cm)
except ValueError as exc:
    print("Expected validation error:", exc)

del request["port"]

assert validate_server_config(cm)

Expected validation error: port must be an integer from 1 to 65535


# 9. Protecting low-priority defaults

Sometimes defaults should be readable but not directly writable.

Python's `MappingProxyType` can expose a read-only view of a dictionary.

Combined with a writable first mapping, this gives a useful pattern:

`mutable overrides -> immutable defaults`

## Problem 9 — Build a protected configuration stack

Requirements:

1. defaults must be immutable through the configuration interface;
2. overrides must remain writable;
3. assigning through the `ChainMap` must not change defaults;
4. changing the original backing dictionary should still be visible through the proxy.

### Step 1: create the proxy

In [39]:
raw_defaults = {
    "theme": "light",
    "timeout": 30,
}

readonly_defaults = MappingProxyType(raw_defaults)
overrides = {}

config = ChainMap(overrides, readonly_defaults)

### Step 2: write an override

In [40]:
config["theme"] = "dark"

assert config["theme"] == "dark"
assert raw_defaults["theme"] == "light"
assert overrides["theme"] == "dark"

### Step 3: prove the proxy rejects direct writes

In [41]:
try:
    readonly_defaults["timeout"] = 10
except TypeError as exc:
    print("Expected:", exc)

Expected: 'mappingproxy' object does not support item assignment


### Step 4: remember that a mapping proxy is still a live view

If the backing dictionary changes, the proxy reflects that change.

In [42]:
raw_defaults["timeout"] = 45

assert readonly_defaults["timeout"] == 45
assert config["timeout"] == 45

print(dict(config))

{'theme': 'dark', 'timeout': 45}


# 10. What if we want different write behavior?

Standard `ChainMap` always writes to the first mapping.

But sometimes we want this policy:

> If the key already exists, update the first mapping that owns it.  
> If it does not exist anywhere, create it in the first mapping.

That requires custom behavior.

## Problem 10 — Implement a write-through `ChainMap`

Create a subclass called `WriteThroughChainMap`.

Rules:

- lookup stays unchanged;
- assignment searches mappings from left to right;
- if the key exists, update the first mapping containing it;
- otherwise insert into the first mapping;
- deletion searches mappings and deletes the first occurrence.

### Solution — override only mutation methods

The inherited lookup behavior is already correct, so we do not need to rewrite it.

In [43]:
class WriteThroughChainMap(ChainMap):
    def __setitem__(self, key, value):
        for mapping in self.maps:
            if key in mapping:
                mapping[key] = value
                return

        self.maps[0][key] = value

    def __delitem__(self, key):
        for mapping in self.maps:
            if key in mapping:
                del mapping[key]
                return

        raise KeyError(key)

Let's compare standard and write-through behavior.

In [44]:
first = {"local_only": 1}
second = {"timeout": 30}
third = {"theme": "light"}

standard = ChainMap(first.copy(), second.copy(), third.copy())
deep = WriteThroughChainMap(first.copy(), second.copy(), third.copy())

standard["timeout"] = 5
deep["timeout"] = 5

print("Standard:")
pprint(standard.maps)

print("\nWrite-through:")
pprint(deep.maps)

Standard:
[{'local_only': 1, 'timeout': 5}, {'timeout': 30}, {'theme': 'light'}]

Write-through:
[{'local_only': 1}, {'timeout': 5}, {'theme': 'light'}]


In the standard `ChainMap`, `timeout` was inserted into the first map.

In the custom version, the second map was edited in place.

In [45]:
assert "timeout" in standard.maps[0]
assert "timeout" not in deep.maps[0]
assert deep.maps[1]["timeout"] == 5

# 11. Auditing what an override layer actually changes

Suppose we have:

- a base configuration,
- an override mapping,
- a combined `ChainMap`.

We may want to answer:

> Which override keys actually change the visible value, and which merely repeat the inherited value?

## Problem 11 — Classify overrides

Write:

```python
def classify_overrides(cm):
    ...
```

Assume:

- `cm.maps[0]` is the override layer;
- `cm.parents` represents inherited configuration.

Return:

```python
{
    "changed": {...},
    "redundant": {...},
    "new": {...}
}
```

Where:

- `changed`: key exists in parents but has a different value;
- `redundant`: key exists in parents with the same value;
- `new`: key does not exist in parents.

### Step 1: build an example

In [46]:
base = {
    "timeout": 30,
    "theme": "light",
    "retries": 3,
}

overrides = {
    "timeout": 5,
    "theme": "light",
    "debug": True,
}

cm = ChainMap(overrides, base)

Here:

- `timeout` is changed,
- `theme` is redundant,
- `debug` is new.

Now let's implement that reasoning.

In [47]:
def classify_overrides(cm):
    current = cm.maps[0]
    inherited = cm.parents

    result = {
        "changed": {},
        "redundant": {},
        "new": {},
    }

    for key, value in current.items():
        if key not in inherited:
            result["new"][key] = value
        elif inherited[key] == value:
            result["redundant"][key] = value
        else:
            result["changed"][key] = {
                "override": value,
                "inherited": inherited[key],
            }

    return result

Let's inspect the classification.

In [48]:
classification = classify_overrides(cm)
pprint(classification)

{'changed': {'timeout': {'inherited': 30, 'override': 5}},
 'new': {'debug': True},
 'redundant': {'theme': 'light'}}


In [49]:
assert "timeout" in classification["changed"]
assert classification["redundant"] == {"theme": "light"}
assert classification["new"] == {"debug": True}

print("Classification checks passed.")

Classification checks passed.


# 12. A transactional way to think about child mappings

An empty child mapping can act like a change-set.

The parent remains untouched until we explicitly decide to copy selected changes into it.

This gives us a lightweight transaction-like workflow:

1. begin with an empty overlay;
2. make experimental changes;
3. inspect them;
4. either discard them or commit them.

## Problem 12 — Guided transaction overlay

Write three functions:

```python
begin(base)
commit(tx)
rollback(tx)
```

For this exercise:

- `begin(base)` returns `ChainMap({}, base)`;
- assignments are recorded in the overlay;
- `commit(tx)` copies the overlay into the base and clears the overlay;
- `rollback(tx)` clears the overlay without changing the base.

We will treat only assignments/updates here, not deletion tombstones.

### Step 1: begin a transaction

In [50]:
def begin(base):
    return ChainMap({}, base)

In [51]:
database_state = {
    "balance": 100,
    "status": "open",
}

tx = begin(database_state)

assert tx.maps[0] == {}
assert tx.maps[1] is database_state

### Step 2: make changes

These should be visible through the transaction but not yet present in the base dictionary.

In [52]:
tx["balance"] = 125
tx["note"] = "manual adjustment"

assert tx["balance"] == 125
assert database_state["balance"] == 100
assert "note" not in database_state

pprint(tx.maps)

[{'balance': 125, 'note': 'manual adjustment'},
 {'balance': 100, 'status': 'open'}]


### Step 3: implement commit

In [53]:
def commit(tx):
    if len(tx.maps) < 2:
        raise ValueError("Transaction must have an overlay and a base mapping")

    overlay = tx.maps[0]
    base = tx.maps[1]

    base.update(overlay)
    overlay.clear()

    return base

Commit the current transaction.

In [54]:
commit(tx)

assert database_state["balance"] == 125
assert database_state["note"] == "manual adjustment"
assert tx.maps[0] == {}

pprint(database_state)

{'balance': 125, 'note': 'manual adjustment', 'status': 'open'}


### Step 4: implement rollback

In [55]:
def rollback(tx):
    if not tx.maps:
        raise ValueError("Invalid transaction")

    tx.maps[0].clear()
    return tx

Start another transaction, make changes, and roll them back.

In [56]:
tx2 = begin(database_state)

tx2["balance"] = 999
tx2["status"] = "closed"

assert tx2["balance"] == 999

rollback(tx2)

assert tx2["balance"] == 125
assert tx2["status"] == "open"
assert database_state["balance"] == 125
assert database_state["status"] == "open"

print("Rollback restored the inherited view.")

Rollback restored the inherited view.


# Final Challenge — A layered application settings manager

Now combine the ideas from the notebook.

We want an object that manages:

- immutable defaults,
- environment values,
- user values,
- request overrides,
- temporary child scopes,
- validation,
- provenance.

The important point is not the amount of code.

The important point is that each feature follows directly from the behaviors we observed earlier.

## Final Problem

Implement `SettingsManager` with:

- `get(key)`
- `set_request(key, value)`
- `reset_request(key)`
- `push_temporary()`
- `pop_temporary()`
- `explain(key)`
- `snapshot()`

Use the precedence:

`temporary scopes -> request -> user -> environment -> defaults`

Defaults should be exposed through `MappingProxyType`.

### Solution — Step 1: construct the permanent chain

We keep the permanent request/user/environment/default layers together.

Temporary scopes will be added in front later.

In [57]:
class SettingsManager:
    def __init__(self, defaults, environment=None, user=None, request=None):
        self._defaults_backing = dict(defaults)
        self._defaults = MappingProxyType(self._defaults_backing)

        self.environment = {} if environment is None else environment
        self.user = {} if user is None else user
        self.request = {} if request is None else request

        self.chain = ChainMap(
            self.request,
            self.user,
            self.environment,
            self._defaults,
        )

        self._temporary_depth = 0

    def get(self, key):
        return self.chain[key]

    def set_request(self, key, value):
        self.request[key] = value

    def reset_request(self, key):
        if key not in self.request:
            raise KeyError(f"{key!r} is not set in request overrides")
        del self.request[key]

    def push_temporary(self):
        self.chain = self.chain.new_child()
        self._temporary_depth += 1

    def pop_temporary(self):
        if self._temporary_depth == 0:
            raise RuntimeError("No temporary scope to pop")
        self.chain = self.chain.parents
        self._temporary_depth -= 1

    def set_temporary(self, key, value):
        if self._temporary_depth == 0:
            raise RuntimeError("Push a temporary scope first")
        self.chain[key] = value

    def explain(self, key):
        return trace_lookup(self.chain, key)

    def snapshot(self):
        return dict(self.chain)

### Step 2: construct a realistic manager

In [58]:
manager = SettingsManager(
    defaults={
        "host": "api.example.com",
        "port": 443,
        "timeout": 30,
        "theme": "light",
    },
    environment={
        "timeout": 20,
    },
    user={
        "theme": "dark",
    },
    request={
        "timeout": 5,
    },
)

assert manager.get("host") == "api.example.com"
assert manager.get("timeout") == 5
assert manager.get("theme") == "dark"

pprint(manager.snapshot())

{'host': 'api.example.com', 'port': 443, 'theme': 'dark', 'timeout': 5}


### Step 3: explain where a value came from

In [59]:
pprint(manager.explain("timeout"))

{'key': 'timeout', 'present_in': [0, 2, 3], 'source_index': 0, 'value': 5}


### Step 4: add a temporary scope

A temporary scope should override even request-level values.

In [60]:
manager.push_temporary()
manager.set_temporary("timeout", 1)
manager.set_temporary("debug", True)

assert manager.get("timeout") == 1
assert manager.get("debug") is True

pprint(manager.snapshot())

{'debug': True,
 'host': 'api.example.com',
 'port': 443,
 'theme': 'dark',
 'timeout': 1}


### Step 5: discard the temporary scope

The request-level value should reappear.

In [61]:
manager.pop_temporary()

assert manager.get("timeout") == 5

try:
    manager.get("debug")
except KeyError:
    print("Temporary debug flag is gone, as expected.")

Temporary debug flag is gone, as expected.


### Step 6: reset a request override

Removing the request's `timeout` should reveal the environment's value.

In [62]:
manager.reset_request("timeout")

assert manager.get("timeout") == 20

pprint(manager.explain("timeout"))

{'key': 'timeout', 'present_in': [2, 3], 'source_index': 2, 'value': 20}


# Extra guided exercises

Try these without looking back at the earlier solutions.

### Exercise A

Write `all_values(cm, key)` that returns all values associated with a key, from highest to lowest priority.

### Exercise B

Write `shadowed_keys(cm)` that returns every key appearing in at least two mappings.

### Exercise C

Write `source_name(cm, key, names)` where `names` is a list such as:

```python
["request", "user", "environment", "defaults"]
```

Return both the visible value and the human-readable source name.

### Exercise D

Create a context manager that automatically pushes a child `ChainMap` scope on entry and removes it on exit.

### Exercise E

Create a configuration report that prints only keys whose effective value differs from the defaults.

### Exercise F

Extend `SettingsManager` with a method:

```python
changed_from_defaults()
```

that returns a normal dictionary containing only effective values that differ from the default layer.

### Exercise G

Implement a custom `ChainMap` subclass that forbids writes to protected keys such as `"host"` and `"port"`.

### Exercise H

Implement a method that reorders `cm.maps` and demonstrate how precedence changes immediately without copying the mappings.

# Closing observations

The most important ideas to keep in mind are:

- a `ChainMap` combines mappings without eagerly copying them;
- lookups search from the first mapping toward the last;
- collisions are resolved by the first matching key;
- writes and deletes normally affect only the first mapping;
- removing a child value may reveal a parent value;
- underlying mappings remain live and can change independently;
- `new_child()` and `parents` make nested scopes natural;
- `.maps` is useful for debugging, provenance, and advanced custom behavior;
- custom subclasses can change mutation policy while preserving lookup semantics.

The advanced patterns in this notebook—configuration stacks, provenance reports, temporary scopes, protected defaults, and transaction-like overlays—are all extensions of those core rules.